In [1]:
import sys
import logging
import json
from pathlib import Path
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, str(Path.cwd().parent / "src"))

# Spotipy/urllib3 Stderr-Noise bei 429 unterdruecken (bringt nur Verwirrung,
# unsere eigene SpotifyRateLimitError reicht).
logging.getLogger("spotipy").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)

%load_ext autoreload
%autoreload 2

from spotify_scraper import (
    get_spotify_client,
    get_artist_albums,
    prune_cache,
    SpotifyRateLimitError,
)
from cover_downloader import download_all_covers

sp = get_spotify_client()
print("Spotify-Client initialisiert")

Spotify-Client initialisiert


In [2]:
# artists.json laden und in DataFrame flatten. Jede Zeile = 1 Artist mit ID.
artists_json_path = Path.cwd().parent / "src" / "artists.json"
with open(artists_json_path, "r", encoding="utf-8") as f:
    genre_artists = json.load(f)

rows = []
missing_ids = []
for genre, entries in genre_artists.items():
    for entry in entries:
        if not entry.get("id"):
            missing_ids.append((genre, entry["name"]))
            continue
        rows.append({
            "genre": genre,
            "artist_name": entry["name"],
            "artist_id": entry["id"],
        })

artists_df = pd.DataFrame(rows)
print(f"{len(artists_df)} Artists mit ID")
print(artists_df.groupby("genre").size().sort_values(ascending=False))

if missing_ids:
    print(f"\n{len(missing_ids)} Artists ohne ID — werden uebersprungen:")
    for genre, name in missing_ids:
        print(f"   [{genre}] {name}")

200 Artists mit ID
genre
alternative_rock    20
classical           20
country             20
hiphop              20
house               20
indie_rock          20
jazz                20
metal               20
reggae              20
techno              20
dtype: int64


In [3]:
# Cache-Hygiene: alte Eintraege aus frueheren Artist-Sets entfernen.
# Behaelt nur Cache-Daten fuer Artists, die jetzt noch in artists.json stehen.
removed = prune_cache(set(artists_df["artist_id"]))
print(f"Cache aufgeraeumt: {len(removed)} alte Eintraege entfernt")

Cache aufgeraeumt: 0 alte Eintraege entfernt


In [ ]:
# Pro Artist alle Alben holen. Ergebnisse werden in data/spotify_cache.json gecached.
# Bei Daily-Cap ueberspringen wir betroffene Artists (per-artist try/except) —
# beim naechsten Lauf werden sie nachgezogen.
# Wenn ALLE rate-limited sind (= 0 Alben gesammelt), brechen wir hart ab.
all_albums = []
rate_limited = []  # (genre, artist_name)

for _, artist_row in tqdm(artists_df.iterrows(), total=len(artists_df), desc="Alben abrufen"):
    try:
        albums = get_artist_albums(sp, artist_row["artist_id"])
    except SpotifyRateLimitError:
        rate_limited.append((artist_row["genre"], artist_row["artist_name"]))
        continue
    for album in albums:
        album["genre"] = artist_row["genre"]
        album["artist_name"] = artist_row["artist_name"]
        all_albums.append(album)

albums_df = pd.DataFrame(all_albums)

if albums_df.empty:
    raise RuntimeError(
        f"Keine Alben gesammelt — alle {len(rate_limited)} Artists rate-limited. "
        f"Daily-Cap noch aktiv. Spaeter erneut versuchen."
    )

print(f"\n{len(albums_df)} Alben insgesamt gesammelt ({len(artists_df) - len(rate_limited)}/{len(artists_df)} Artists)")
print(f"   Unique Album-IDs: {albums_df['album_id'].nunique()}")
print(f"   (Differenz = Alben, die mehreren Artists zugeordnet werden, z.B. Collabs)")
if rate_limited:
    print(f"\n{len(rate_limited)} Artists Rate-Limited — Notebook spaeter erneut starten:")
    for genre, name in rate_limited[:10]:
        print(f"   [{genre}] {name}")
    if len(rate_limited) > 10:
        print(f"   ... und {len(rate_limited) - 10} weitere")

In [5]:
genre_stats = albums_df.groupby("genre").agg(
    anzahl_alben=("album_id", "nunique"),
    anzahl_artists=("artist_id", "nunique"),
).sort_values("anzahl_alben", ascending=False)

print(genre_stats)

KeyError: 'genre'

In [ ]:
data_dir = Path.cwd().parent / "data"
data_dir.mkdir(exist_ok=True)

artists_df.to_csv(data_dir / "artists_raw.csv", index=False)
albums_df.to_csv(data_dir / "albums_raw.csv", index=False)

print(f"Gespeichert:")
print(f"   {data_dir / 'artists_raw.csv'} ({len(artists_df)} Zeilen)")
print(f"   {data_dir / 'albums_raw.csv'} ({len(albums_df)} Zeilen)")

In [ ]:
# Cover-Bilder runterladen nach data/covers/{genre}/{album_id}.jpg.
# Globale Deduplizierung nach album_id (Collabs landen nur in einem Genre).
# Resume-fähig: bestehende Dateien werden uebersprungen.
download_all_covers()

In [ ]:
# Sanity-Check: tatsaechlich vorhandene Cover pro Genre auf der Disk zaehlen.
# Hilft Class-Imbalance vorm Training zu erkennen.
covers_dir = Path.cwd().parent / "data" / "covers"
counts = {
    g.name: len(list(g.glob("*.jpg")))
    for g in sorted(covers_dir.iterdir())
    if g.is_dir()
}
total = sum(counts.values())

print(f"Cover auf Disk (gesamt: {total})")
for genre, n in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "█" * int(n / max(counts.values()) * 30) if counts.values() else ""
    print(f"   {genre:<20s} {n:>4d}  {bar}")

In [ ]:
# Visual Sanity-Check: 3 Zufalls-Cover pro Genre als Grid anzeigen.
# Schneller Eyeball-Check ob die Genre-Cover plausibel aussehen vor Training.
import random
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)

genres_with_covers = sorted([
    g for g in covers_dir.iterdir() if g.is_dir() and any(g.glob("*.jpg"))
])

if not genres_with_covers:
    print("Keine Cover gefunden — erst download_all_covers() ausfuehren.")
else:
    n_samples = 3
    fig, axes = plt.subplots(
        len(genres_with_covers), n_samples,
        figsize=(n_samples * 2.5, len(genres_with_covers) * 2.5),
    )
    if len(genres_with_covers) == 1:
        axes = axes.reshape(1, -1)

    for row, genre_dir in enumerate(genres_with_covers):
        files = list(genre_dir.glob("*.jpg"))
        sample = random.sample(files, min(n_samples, len(files)))
        for col in range(n_samples):
            ax = axes[row, col]
            ax.set_xticks([])
            ax.set_yticks([])
            if col < len(sample):
                ax.imshow(Image.open(sample[col]))
        axes[row, 0].set_ylabel(
            genre_dir.name, rotation=0, labelpad=70,
            fontsize=11, va="center", ha="right",
        )

    plt.tight_layout()
    plt.show()